## Observations

1. Missing values in Download_Speed_Mbps were replaced using the median value.
2. Missing values in Signal_Strength_dBm were replaced using the median value.
3. Duplicate records were successfully removed.
4. Timestamp column was converted into datetime format.
5. The cleaned dataset was exported as campus_wifi_cleaned.csv.
6. The cleaned dataset is now ready for SQLite database integration and SQL analysis.

In [ ]:
import sqlite3
import pandas as pd
print("Libraries Imported Successfully")

df_sql = pd.read_csv(
    "campus_wifi_cleaned.csv"
)
print("Dataset Loaded Successfully")
print(f"Shape: {df_sql.shape}")

Libraries Imported Successfully
Dataset Loaded Successfully
Shape: (10000, 15)


In [ ]:
conn = sqlite3.connect(
    "campus_wifi.db"
)
print("Database Created Successfully")

Database Created Successfully


In [ ]:
df_sql.to_sql(
    "wifi_logs",
    conn,
    if_exists="replace",
    index=False
)
print("Data Loaded Into SQLite Successfully")

Data Loaded Into SQLite Successfully


In [ ]:
verification_query = """
SELECT COUNT(*) AS total_records
FROM wifi_logs
"""
verification_df = pd.read_sql_query(
    verification_query,
    conn
)
display(verification_df)

,total_records
0,10000


In [ ]:
cursor = conn.cursor()
cursor.execute(
    "PRAGMA table_info(wifi_logs)"
)
columns = cursor.fetchall()

print("="*60)
print("TABLE SCHEMA")
print("="*60)

for col in columns:
    print(col)

TABLE SCHEMA
(0, 'Record_ID', 'INTEGER', 0, None, 0)
(1, 'Timestamp', 'TEXT', 0, None, 0)
(2, 'Building', 'TEXT', 0, None, 0)
(3, 'Floor', 'TEXT', 0, None, 0)
(4, 'Zone', 'TEXT', 0, None, 0)
(5, 'Connected_Users', 'INTEGER', 0, None, 0)
(6, 'Bandwidth_Usage_MB', 'REAL', 0, None, 0)
(7, 'Download_Speed_Mbps', 'REAL', 0, None, 0)
(8, 'Upload_Speed_Mbps', 'REAL', 0, None, 0)
(9, 'Latency_ms', 'REAL', 0, None, 0)
(10, 'Packet_Loss_Percent', 'REAL', 0, None, 0)
(11, 'Signal_Strength_dBm', 'REAL', 0, None, 0)
(12, 'Weather', 'TEXT', 0, None, 0)
(13, 'Event_Occurring', 'TEXT', 0, None, 0)
(14, 'Congestion_Status', 'TEXT', 0, None, 0)


In [ ]:
query = """
SELECT DISTINCT Congestion_Status
FROM wifi_logs
"""
congestion_levels = pd.read_sql_query(
    query,
    conn
)

display(congestion_levels)

,Congestion_Status
0,Medium
1,Low
2,High


In [ ]:
test_query = """
SELECT Building,
COUNT(*) AS total_records
FROM wifi_logs
GROUP BY Building
"""
test_result = pd.read_sql_query(
    test_query,
    conn
)

display(test_result)

,Building,total_records
0,AI_BLOCK,3093
1,AMENITY,647
2,BUS_AREA,293
3,HOSTEL_BOYS,283
4,HOSTEL_GIRLS,294
5,MAIN_BLOCK,2972
6,MARIO,306
7,MECH_BLOCK,2112


## Observations

1. A SQLite database named campus_wifi.db was created successfully.
2. The cleaned dataset was loaded into the table wifi_logs.
3. All records were inserted successfully.
4. The table schema was verified using PRAGMA.
5. Distinct buildings and congestion levels were retrieved successfully.
6. SQLite is now ready for performing SQL analysis and transformations.

# Section 4: SQL Analysis

SQL (Structured Query Language) is used to analyze data stored in the SQLite database.

The following SQL operations are demonstrated:

1. Filtering
2. Sorting
3. Aggregation
4. Group By
5. Conditional Queries

These analyses help identify congestion patterns, network performance trends, and usage behavior across the campus WiFi infrastructure.

In [ ]:
# =====================================================
# SQL ANALYSIS QUERIES
# =====================================================

# 1. FILTERING
filter_query = """
SELECT *
FROM wifi_logs
WHERE Connected_Users > 180
"""

filter_df = pd.read_sql_query(filter_query, conn)


# 2. SORTING
sort_query = """
SELECT Building,
       Zone,
       Download_Speed_Mbps
FROM wifi_logs
ORDER BY Download_Speed_Mbps DESC
LIMIT 10
"""

sort_df = pd.read_sql_query(sort_query, conn)


# 3. AGGREGATION
aggregation_query = """
SELECT
AVG(Latency_ms) AS Average_Latency
FROM wifi_logs
"""

aggregation_df = pd.read_sql_query(aggregation_query, conn)


# 4. GROUP BY
group_query = """
SELECT
Building,
AVG(Connected_Users) AS Avg_Users
FROM wifi_logs
GROUP BY Building
ORDER BY Avg_Users DESC
"""

group_df = pd.read_sql_query(group_query, conn)


# 5. CONDITIONAL QUERY
condition_query = """
SELECT *
FROM wifi_logs
WHERE Event_Occurring != 'No_Event'
AND Congestion_Status = 'High'
"""

condition_df = pd.read_sql_query(condition_query, conn)

print("All SQL Queries Executed Successfully!")

All SQL Queries Executed Successfully!


In [ ]:
print("="*70)
print("1. FILTERING QUERY")
print("Connected Users > 180")
print("="*70)
display(filter_df.head())

print("="*70)
print("2. SORTING QUERY")
print("Top 10 Highest Download Speeds")
print("="*70)
display(sort_df)

print("="*70)
print("3. AGGREGATION QUERY")
print("Average Latency")
print("="*70)
display(aggregation_df)

print("="*70)
print("4. GROUP BY QUERY")
print("Average Users by Building")
print("="*70)
display(group_df)

print("="*70)
print("5. CONDITIONAL QUERY")
print("High Congestion During Events")
print("="*70)
display(condition_df.head())

1. FILTERING QUERY
Connected Users > 180


,Record_ID,Timestamp,Building,Floor,Zone,Connected_Users,Bandwidth_Usage_MB,Download_Speed_Mbps,Upload_Speed_Mbps,Latency_ms,Packet_Loss_Percent,Signal_Strength_dBm,Weather,Event_Occurring,Congestion_Status
0,12,2025-07-17 18:46:00,AMENITY,GROUND,Food Court,183,1202.68,49.35,29.25,126.95,4.78,-57.0,Sunny,Culturals,High
1,15,2025-10-13 15:57:00,MAIN_BLOCK,FLOOR1,Auditorium,254,2047.03,10.30,11.50,173.10,5.73,-59.0,Sunny,Freshwarite,High
2,19,2025-04-26 03:58:00,AI_BLOCK,FLOOR2,Vista Hall,266,2932.10,8.00,8.50,180.90,6.63,-58.0,Rainy,Thiran,High
3,23,2025-09-13 20:52:00,AI_BLOCK,FLOOR2,Vista Hall,293,3960.69,8.00,5.00,198.45,7.00,-75.0,Cloudy,Thiran,High
4,29,2025-09-27 14:15:00,MAIN_BLOCK,GROUND,Classroom,192,1431.29,44.40,27.00,132.80,6.14,-61.0,Sunny,Culturals,High


2. SORTING QUERY
Top 10 Highest Download Speeds


,Building,Zone,Download_Speed_Mbps
0,AMENITY,Food Court,128.0
1,MECH_BLOCK,Lab,128.0
2,AMENITY,Food Court,128.0
3,MECH_BLOCK,Lab,128.0
4,MECH_BLOCK,Lab,128.0
5,BUS_AREA,Bus Bay,128.0
6,MECH_BLOCK,CAD Lab,128.0
7,MAIN_BLOCK,Classroom,128.0
8,AMENITY,Music Room,128.0
9,AI_BLOCK,ML Lab,128.0


3. AGGREGATION QUERY
Average Latency


,Average_Latency
0,81.6099


4. GROUP BY QUERY
Average Users by Building


,Building,Avg_Users
0,AMENITY,135.074189
1,AI_BLOCK,120.782089
2,HOSTEL_GIRLS,117.598639
3,HOSTEL_BOYS,114.780919
4,MAIN_BLOCK,111.343540
5,MECH_BLOCK,104.052083
6,MARIO,100.323529
7,BUS_AREA,78.706485


5. CONDITIONAL QUERY
High Congestion During Events


,Record_ID,Timestamp,Building,Floor,Zone,Connected_Users,Bandwidth_Usage_MB,Download_Speed_Mbps,Upload_Speed_Mbps,Latency_ms,Packet_Loss_Percent,Signal_Strength_dBm,Weather,Event_Occurring,Congestion_Status
0,12,2025-07-17 18:46:00,AMENITY,GROUND,Food Court,183,1202.68,49.35,29.25,126.95,4.78,-57.0,Sunny,Culturals,High
1,15,2025-10-13 15:57:00,MAIN_BLOCK,FLOOR1,Auditorium,254,2047.03,10.30,11.50,173.10,5.73,-59.0,Sunny,Freshwarite,High
2,19,2025-04-26 03:58:00,AI_BLOCK,FLOOR2,Vista Hall,266,2932.10,8.00,8.50,180.90,6.63,-58.0,Rainy,Thiran,High
3,23,2025-09-13 20:52:00,AI_BLOCK,FLOOR2,Vista Hall,293,3960.69,8.00,5.00,198.45,7.00,-75.0,Cloudy,Thiran,High
4,29,2025-09-27 14:15:00,MAIN_BLOCK,GROUND,Classroom,192,1431.29,44.40,27.00,132.80,6.14,-61.0,Sunny,Culturals,High


## Observations

### Filtering
The filtering query identified locations where the number of connected users exceeded 180, indicating highly crowded network zones.

### Sorting
The sorting query displayed the locations with the highest download speeds, helping identify well-performing network areas.

### Aggregation
The average latency of the campus network was calculated to understand overall network responsiveness.

### Group By
Average user counts were analyzed for each building to determine which locations experience the highest WiFi demand.

### Conditional Query
The conditional query identified locations experiencing high congestion during campus events such as Thiran, Culturals, FlashMob, and Freshwarite.

# Section 5: SQL-Based Data Transformation

Data transformation was performed using SQL CASE statements.

Two new business-oriented columns were created:

1. Traffic_Level
   - Light
   - Moderate
   - Heavy

2. Network_Health
   - Excellent
   - Good
   - Poor

The transformed dataset was then exported for further analysis.

In [ ]:
transformation_query = """
CREATE TABLE wifi_logs_transformed AS

SELECT *,

CASE
    WHEN Connected_Users < 100 THEN 'Light'
    WHEN Connected_Users < 180 THEN 'Moderate'
    ELSE 'Heavy'
END AS Traffic_Level,

CASE
    WHEN Latency_ms < 60 THEN 'Excellent'
    WHEN Latency_ms < 100 THEN 'Good'
    ELSE 'Poor'
END AS Network_Health

FROM wifi_logs
"""

conn.execute("DROP TABLE IF EXISTS wifi_logs_transformed")

conn.execute(transformation_query)

conn.commit()

print("Transformed Table Created Successfully!")

Transformed Table Created Successfully!


In [ ]:
transformed_df = pd.read_sql_query(
    """
    SELECT *
    FROM wifi_logs_transformed
    LIMIT 10
    """,
    conn
)

display(transformed_df)

,Record_ID,Timestamp,Building,Floor,Zone,Connected_Users,Bandwidth_Usage_MB,Download_Speed_Mbps,Upload_Speed_Mbps,Latency_ms,Packet_Loss_Percent,Signal_Strength_dBm,Weather,Event_Occurring,Congestion_Status,Traffic_Level,Network_Health
0,1,2025-09-12 01:18:00,AMENITY,GROUND,Music Room,100,1137.17,95.00,50.00,73.00,3.39,-58.0,Rainy,Culturals,Medium,Moderate,Good
1,2,2025-03-02 18:18:00,MECH_BLOCK,GROUND,FABLAB,63,747.05,115.35,59.25,48.95,1.87,-57.0,Sunny,No_Event,Low,Light,Excellent
2,3,2025-05-19 05:21:00,MARIO,GROUND,Snack Corner,115,1323.06,86.75,46.25,82.75,2.81,-58.0,Sunny,No_Event,Medium,Moderate,Good
3,4,2025-04-19 20:10:00,AI_BLOCK,FLOOR3,Data Visualization Lab,94,1395.81,98.30,51.50,69.10,3.31,-44.0,Rainy,No_Event,Low,Light,Good
4,5,2025-12-06 22:12:00,MAIN_BLOCK,FLOOR2,IT Project Lab,57,828.78,118.65,60.75,45.05,3.57,-72.0,Sunny,Freshwarite,Low,Light,Excellent
5,6,2025-02-17 07:48:00,AI_BLOCK,FLOOR3,Cyber Lab,99,702.74,95.55,50.25,72.35,3.47,-46.0,Cloudy,No_Event,Low,Light,Good
6,7,2025-11-16 16:17:00,MECH_BLOCK,FLOOR1,Classroom,164,1821.93,59.80,34.00,114.60,3.84,-63.0,Cloudy,Thiran,Medium,Moderate,Poor
7,8,2025-05-21 11:23:00,AI_BLOCK,FLOOR3,Cyber Lab,150,1340.83,67.50,37.50,105.50,4.64,-41.0,Cloudy,No_Event,Medium,Moderate,Poor
8,9,2025-01-17 03:27:00,AI_BLOCK,FLOOR1,Classroom,92,1037.83,99.40,52.00,67.80,4.27,-36.0,Sunny,No_Event,Low,Light,Good
9,10,2025-09-19 17:36:00,AMENITY,GROUND,Music Room,48,621.61,123.60,63.00,39.20,1.86,-57.0,Sunny,Freshwarite,Low,Light,Excellent


In [ ]:
transformed_df_full = pd.read_sql_query(
    """
    SELECT *
    FROM wifi_logs_transformed
    """,
    conn
)

transformed_df_full.to_csv(
    "campus_wifi_transformed.csv",
    index=False
)

print("Transformed Dataset Exported Successfully!")
print("File Name: campus_wifi_transformed.csv")

Transformed Dataset Exported Successfully!
File Name: campus_wifi_transformed.csv


## Observations

1. SQL CASE statements were used to derive new business-oriented features.
2. Traffic_Level categorizes network load into Light, Moderate, and Heavy traffic conditions.
3. Network_Health categorizes network quality based on latency measurements.
4. The transformed dataset was exported successfully for further ETL, PySpark, and Machine Learning tasks.